# Flipkart AI Customer Support Assistant

## Policy Bound Context Engineering for Refund Decision Support

### Project Objective

This project demonstrates a professional context engineering workflow for a Flipkart AI Customer Support Assistant.

The system evaluates refund requests using verified customer information, delivery data, and a defined refund policy. It is designed to avoid unsupported assumptions and to request clarification when required information is missing.


## 1. System Guardrails

The system guardrails define how the AI assistant must behave.

The assistant must not guess missing information, invent order details, or make refund decisions without sufficient context.


In [1]:
SYSTEM_CONTEXT = '''
You are a professional Flipkart customer support assistant.

Follow these rules strictly:

1. Answer only the customer's refund question.
2. Use only the information provided in the context.
3. Do not assume or invent missing information.
4. If the delivery date is missing, ask the customer to provide the delivery date.
5. Do not approve or reject a refund when the delivery date is unavailable.
6. Apply the supplied refund policy exactly.
7. Keep the response concise, formal, and professional.
'''


## 2. Refund Policy

For this educational demonstration, refund eligibility is determined using the following policy.


In [2]:
REFUND_POLICY = '''
Refund Policy for this demonstration:

A refund is allowed only when the request is made within 7 days of delivery.

The delivery date must be available before refund eligibility can be determined.
'''


## 3. Incomplete Context Scenario

The customer asks whether an order is eligible for a refund.

The order ID is available, but the delivery date is missing. Therefore, the assistant must request the delivery date instead of making a refund decision.


In [3]:
user_query_incomplete = "Can I get a refund for order FK10245?"

customer_context_incomplete = {
    "customer_type": "registered_customer",
    "order_id": "FK10245",
    "delivery_date": None
}

print("Customer Question:", user_query_incomplete)
print("Customer Context:", customer_context_incomplete)


Customer Question: Can I get a refund for order FK10245?
Customer Context: {'customer_type': 'registered_customer', 'order_id': 'FK10245', 'delivery_date': None}


## 4. Assemble the Incomplete Context


In [4]:
final_prompt_incomplete = f'''
Company Policy:
{REFUND_POLICY}

Customer Information:
Customer Type: {customer_context_incomplete["customer_type"]}
Order ID: {customer_context_incomplete["order_id"]}
Delivery Date: {customer_context_incomplete["delivery_date"]}

Customer Question:
{user_query_incomplete}

Required response:
If the delivery date is missing, ask only for the delivery date needed to evaluate refund eligibility.
Do not approve or reject the refund.
'''

print(final_prompt_incomplete)



Company Policy:

Refund Policy for this demonstration:

A refund is allowed only when the request is made within 7 days of delivery.

The delivery date must be available before refund eligibility can be determined.


Customer Information:
Customer Type: registered_customer
Order ID: FK10245
Delivery Date: None

Customer Question:
Can I get a refund for order FK10245?

Required response:
If the delivery date is missing, ask only for the delivery date needed to evaluate refund eligibility.
Do not approve or reject the refund.



## 5. Secure API Configuration

The API key is loaded securely from Google Colab Secrets.

Create a Colab secret named api_key before running this section. The actual API key must never be written directly inside the notebook or committed to GitHub.


In [5]:


from google.colab import userdata
from openai import OpenAI

MY_API_KEY = userdata.get("api_key")

if not MY_API_KEY:
    raise ValueError("API key not found. Add a Colab Secret named api_key.")

client = OpenAI(
    api_key=MY_API_KEY,
    base_url="https://nexusapi.navigatelabs.ai"
)

print("API configuration completed securely.")


API configuration completed securely.


## 6. Evaluate the Incomplete Context

The model is now asked to respond to the customer's refund request using only the available context.


In [7]:
response_incomplete = client.chat.completions.create(
    model="gpt-4.1-nano",
    messages=[
        {
            "role": "system",
            "content": SYSTEM_CONTEXT
        },
        {
            "role": "user",
            "content": final_prompt_incomplete
        }
    ],
    temperature=0
)

print(response_incomplete.choices[0].message.content)


Please provide the delivery date for order FK10245 to evaluate your refund request.


## Expected Outcome

Customer request:

Can I get a refund for order FK10245?

Expected professional response:

Please provide the delivery date of order FK10245 so I can check whether it is eligible for a refund under the 7 day refund policy.

The assistant should not approve or reject the refund because the required delivery date is unavailable.


## 7. Policy Conflict Scenario

The customer asks the same refund question again.

In this scenario, the delivery date is available. The system calculates the number of days since delivery and evaluates the request against the same 7 day refund policy.


In [8]:
from datetime import datetime

user_query_conflict = "Can I get a refund for order FK10245?"

customer_context_conflict = {
    "customer_type": "registered_customer",
    "order_id": "FK10245",
    "delivery_date": "2026-08-30"
}

current_date = datetime(2026, 9, 11)

delivery_date = datetime.strptime(
    customer_context_conflict["delivery_date"],
    "%Y-%m-%d"
)

days_since_delivery = (current_date - delivery_date).days

print("Customer Question:", user_query_conflict)
print("Delivery Date:", customer_context_conflict["delivery_date"])
print("Days Since Delivery:", days_since_delivery)


Customer Question: Can I get a refund for order FK10245?
Delivery Date: 2026-08-30
Days Since Delivery: 12


## 8. Assemble the Policy Evaluation Context


In [9]:
final_prompt_conflict = f'''
Company Policy:
{REFUND_POLICY}

Customer Information:
Customer Type: {customer_context_conflict["customer_type"]}
Order ID: {customer_context_conflict["order_id"]}
Delivery Date: {customer_context_conflict["delivery_date"]}
Current Date: {current_date.strftime("%Y-%m-%d")}
Days Since Delivery: {days_since_delivery}

Customer Question:
{user_query_conflict}

Required response:
Answer only whether this order is eligible for a refund under the supplied policy.
Use the calculated days since delivery.
State the reason clearly and professionally.
'''
print(final_prompt_conflict)



Company Policy:

Refund Policy for this demonstration:

A refund is allowed only when the request is made within 7 days of delivery.

The delivery date must be available before refund eligibility can be determined.


Customer Information:
Customer Type: registered_customer
Order ID: FK10245
Delivery Date: 2026-08-30
Current Date: 2026-09-11
Days Since Delivery: 12

Customer Question:
Can I get a refund for order FK10245?

Required response:
Answer only whether this order is eligible for a refund under the supplied policy.
Use the calculated days since delivery.
State the reason clearly and professionally.



## 9. Evaluate the Policy Conflict

The assistant now evaluates the same refund question using the available delivery date and the defined refund policy.


In [11]:
response_conflict = client.chat.completions.create(
    model="gpt-4.1-nano",
    messages=[
        {
            "role": "system",
            "content": SYSTEM_CONTEXT
        },
        {
            "role": "user",
            "content": final_prompt_conflict
        }
    ],
    temperature=0
)

print(response_conflict.choices[0].message.content)


This order was delivered 12 days ago, which exceeds the 7-day refund window specified in our policy. Therefore, it is not eligible for a refund.


## 10. System Workflow

Customer Refund Request

System Guardrails

Refund Policy

Customer Context

Delivery Date Validation

Date Difference Calculation

Policy Evaluation

Professional AI Response


## Conclusion

This project demonstrates a policy bound context engineering approach for an AI customer support assistant.

When the delivery date is missing, the assistant requests the required information without making an unsupported decision.

When the delivery date is available, the system calculates the number of days since delivery and evaluates the refund request against the defined policy.

This approach improves consistency, reliability, and transparency in AI assisted customer support.
